In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from  sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
#load titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

#select relevant features and target
df = df[['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Survived']]

df.fillna({'Age': df['Age'].median(), 'Embarked': df['Embarked'].mode()[0]}, inplace=True)

# define feature  and target
X = df.drop(columns=['Survived'])
y = df['Survived']

#Feature scaling and encoding 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Fare']),
        ('cat', OneHotEncoder(), ['Pclass', 'Sex', 'Embarked'])
    ])

X_processed = preprocessor.fit_transform(X)

#train and evaluate logistic regression model
log_model = LogisticRegression()
log_scores = cross_val_score(log_model, X_processed, y, cv=5, scoring ='accuracy')
print ("Logistic Regression Accuracy:", log_scores.mean())

#train and evaluate random forest model
rf_model = RandomForestClassifier( random_state=42)
rf_scores = cross_val_score(rf_model, X_processed, y, cv=5, scoring ='accuracy')
print ("Random Forest Accuracy:", rf_scores.mean())

#define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10,20],
    'min_samples_split': [2, 5, 10]
}

#perform grid search
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_processed, y)

#display best parameters and score
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Logistic Regression Accuracy: 0.7890088506685079
Random Forest Accuracy: 0.808097420124286
Best Hyperparameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best Cross-Validation Accuracy: 0.8339087314041805
